This notebook compares the performance of the force estimation model across two different target force distributions: **Covariance = 2I** and **Covariance = 4I**.

In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

rep_path = '../../../'

## Processing Logic
The function below handles loading, cleaning, and metric calculation using named columns for stability.

In [ ]:
def clean_and_process_dataset(file_path, gt_range, est_range, force_col_idx, filter_col_idx):
    """
    Loads Excel, calculates errors, filters force, and removes non-consecutive coordinate blocks.
    """
    df = pd.read_excel(file_path, header=0)
    
    # 1. Calculate Absolute Errors and assign to NAMED columns
    # This prevents IndexError if the original file has fewer columns than expected
    errors = (df.iloc[:, gt_range].sub(df.iloc[:, est_range].values)).abs()
    df['Error_X'] = errors.iloc[:, 0]
    df['Error_Y'] = errors.iloc[:, 1]
    df['Error_F'] = errors.iloc[:, 2]
    
    # 2. Filter for Force Presence (Value == 1)
    df = df[df.iloc[:, filter_col_idx] == 1].copy()
    
    # 3. Handle Force Sign and Threshold (<= 2N)
    df.iloc[:, force_col_idx] *= -1
    df = df[df.iloc[:, force_col_idx] <= 2].copy()
    
    # 4. Sort and Clean: Identify blocks of at least 3 consecutive identical coordinates
    # Note: Using first two columns (0, 1) as the sorting key based on previous logic
    df_sorted = df.sort_values(
        by=[df.columns[0], df.columns[1]],
        ascending=[False, False]
    ).reset_index(drop=True)

    rows_to_delete = np.zeros(len(df_sorted), dtype=bool)
    for i in range(len(df_sorted) - 2):
        # Check for 3 consecutive equal values in column 0
        if not (df_sorted.iloc[i, 0] == df_sorted.iloc[i + 1, 0] == df_sorted.iloc[i + 2, 0]):
            rows_to_delete[i:i + 3] = True
        # Check for 3 consecutive equal values in column 1
        if not (df_sorted.iloc[i, 1] == df_sorted.iloc[i + 1, 1] == df_sorted.iloc[i + 2, 1]):
            rows_to_delete[i:i + 3] = True
            
    df_cleaned = df_sorted[~rows_to_delete].copy()
    
    # 5. Calculate Euclidean Distance
    df_cleaned['euclidean_dist'] = np.sqrt(df_cleaned['Error_X']**2 + df_cleaned['Error_Y']**2)
    
    return df_cleaned

def get_regional_summary(df, x_coord_idx, y_coord_idx, dist_name):
    """
    Calculates Mean Euclidean Error and Force MAE for Taxel and Sensor regions.
    """
    # Edge detection logic for 23x23 grid
    is_edge = (df.iloc[:, x_coord_idx] > -0.375) | (df.iloc[:, x_coord_idx] < -17.625) | \
              (df.iloc[:, y_coord_idx] > -0.375) | (df.iloc[:, y_coord_idx] < -17.625)
    
    # Inside Taxels mask
    is_inside_taxel = (~is_edge) & (
        ((df.iloc[:, x_coord_idx] <= -1.5) & (df.iloc[:, x_coord_idx] >= -3.0)) | \
        ((df.iloc[:, x_coord_idx] <= -6.0) & (df.iloc[:, x_coord_idx] >= -7.5)) | \
        ((df.iloc[:, x_coord_idx] <= -10.5) & (df.iloc[:, x_coord_idx] >= -12.0)) | \
        ((df.iloc[:, x_coord_idx] <= -15.0) & (df.iloc[:, x_coord_idx] >= -16.5))
    ) & (
        ((df.iloc[:, y_coord_idx] <= -1.5) & (df.iloc[:, y_coord_idx] >= -3.0)) | \
        ((df.iloc[:, y_coord_idx] <= -6.0) & (df.iloc[:, y_coord_idx] >= -7.5)) | \
        ((df.iloc[:, y_coord_idx] <= -10.5) & (df.iloc[:, y_coord_idx] >= -12.0)) | \
        ((df.iloc[:, y_coord_idx] <= -15.0) & (df.iloc[:, y_coord_idx] >= -16.5))
    )
    
    regions = {"Sensor": ~is_edge}
    
    stats = []
    for name, mask in regions.items():
        subset = df[mask]
        if not subset.empty:
            stats.append({
                "Distribution": dist_name,
                "Region": name,
                "MEE": subset['euclidean_dist'].mean(),
                "MAE_Force": subset['Error_F'].mean()
            })
    return pd.DataFrame(stats)

## Processing Datasets
We process both Covariance 2 and Covariance 4 sets.

In [ ]:
# Covariance 2 Configuration
path_cov2 = os.path.join(rep_path, 'Data/Results/Force_Map_Estimation_Results/Cov_2/Val_Error_Cov2.xlsx')
df_cov2 = clean_and_process_dataset(path_cov2, range(0,3), range(6,9), 2, 3)
summary_cov2 = get_regional_summary(df_cov2, 0, 1, "Covariance 2")

# Covariance 4 Configuration
path_cov4 = os.path.join(rep_path, 'Data/Results/Force_Map_Estimation_Results/Cov_4/Val_Error_Cov4.xlsx')
df_cov4 = clean_and_process_dataset(path_cov4, range(0,3), range(6,9), 2, 3)
summary_cov4 = get_regional_summary(df_cov4, 0, 1, "Covariance 4")

# Merge for Plotting
all_stats = pd.concat([summary_cov2, summary_cov4])

## Final Comparative Visualization

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Spatial Error (MEE in mm)", "Force Error (MAE in N)"))

for dist in ["Covariance 2", "Covariance 4"]:
    subset = all_stats[all_stats['Distribution'] == dist]
    
    # MEE Bar
    fig.add_trace(go.Bar(
        x=subset['Region'], y=subset['MEE'], name=f"{dist} MEE",
        text=[f"{v:.2f}" for v in subset['MEE']], textposition='auto'
    ), row=1, col=1)
    
    # Force MAE Bar
    fig.add_trace(go.Bar(
        x=subset['Region'], y=subset['MAE_Force'], name=f"{dist} Force MAE",
        text=[f"{v:.2f}" for v in subset['MAE_Force']], textposition='auto'
    ), row=1, col=2)

fig.update_layout(
    barmode='group',
    title_text="Comparison of Estimation Errors: Covariance 2 vs Covariance 4 (Validation dataset)",
    template="plotly_white"
)
fig.show()

In [ ]:
# Final results printout for Mean Euclidean Error (MEE) and Force Mean Absolute Error (MAE)
print("FORCE ESTIMATION PERFORMANCE SUMMARY (Validation Dataset)")
for dist in all_stats['Distribution'].unique():
    print(f"\nTarget Distribution: {dist}")
    subset = all_stats[all_stats['Distribution'] == dist]
    for _, row in subset.iterrows():
        print(f"Region: {row['Region']:<15} | MEE: {row['MEE']:.2f} mm | Force MAE: {row['MAE_Force']:.2f} N")